<a href="https://colab.research.google.com/github/Shreenitya/nitya/blob/main/Animation_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import geemap.geemap as map
import ee
ee.Authenticate()
ee.Initialize(project ='shreenitya0428')
m = map.Map()

In [32]:
TamilNadu = ee.FeatureCollection('projects/shreenitya0428/assets/TamilNadu')
m.addLayer(TamilNadu,{},'TamilNadu')
m

Map(bottom=812.0, center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [42]:
# TN_region = ee.Geometry.Polygon(
#     [
#     [76.217, 8.068],
#     [76.217, 13.541],
#     [80.450, 13.541],
#     [80.450, 8.068],
#     [76.217, 8.068],
# ],
#     None,
#     False)

In [101]:
region = ee.Geometry.Polygon(
    [[[74.060547, 7.740675],
     [74.060547, 14.065392],
     [82.964844, 14.065392],
     [82.964844, 7.740675],
     [74.060547, 7.740675]]],
    None,
    False)

In [102]:
ndvi = ee.ImageCollection("MODIS/061/MOD13A2").select('NDVI')

In [103]:
ndvi_Mh = ndvi.map(lambda img: img.set('doy',ee.Date(img.get('system:time_start')).getRelative('day', 'year')));
distinctDOY = ndvi_Mh.filterDate('2023-02-18', '2024-02-18');

In [104]:
filter = ee.Filter.equals(leftField ='doy',rightField = 'doy')
join = ee.Join.saveAll('doy_matches')
joincol = ee.ImageCollection(join.apply(distinctDOY,ndvi_Mh,filter))

In [105]:
composite = joincol.map(lambda img: ee.ImageCollection.fromImages(img.get('doy_matches')).reduce(ee.Reducer.mean()).set('doy',ee.Number(img.get('doy'))))

In [106]:
vis_para = {
  'min': 0,
  'max': 9000,
  'palette': [
    'ffffff', 'ce7e45', 'df923d', 'f1b555', 'fcd163', '99b718', '74a901',
    '66a000', '529400', '3e8601', '207401', '056201', '004c00', '023b01',
    '012e01', '011d01', '011301'
  ],
};

In [107]:
rgbvis= composite.map(lambda img: img.visualize(bands = ['NDVI_mean'],**vis_para).clip(TamilNadu))

In [108]:
gifParams = {
  'region': region,
  'dimensions': 600,
  'crs': 'EPSG:4326',
  'framesPerSecond': 10,
  'format':'gif',
}

In [109]:
print(rgbvis.getVideoThumbURL(gifParams))

https://earthengine.googleapis.com/v1/projects/shreenitya0428/videoThumbnails/df9f2ef36b9477e2eb1554fd43cd709a-8aa680d1125b9222a84cf51b0f1c45ed:getPixels
